<a href="https://colab.research.google.com/github/Sriramkannan1/DL_Learnings/blob/main/Rag_Implementation_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu sentence_transformers openai

In [ ]:
from sentence_transformers import sentence_transformer
import faiss
import openai

In [ ]:
!pip install groq

In [ ]:
from groq import Groq
from google.colab import userdata
groq_api_key = userdata.get('groq_api')


In [ ]:
docs=[
    "Mahendra Singh Dhoni (born 7 July 1981) is an Indian professional cricketer who plays as a right-handed batter and a wicket-keeper. ",
    "Born in Ranchi, Dhoni made his first class debut for Bihar in 1999. He made his debut for the Indian cricket team on 23 December 2004 in an ODI against Bangladesh and played his first test a year later against Sri Lanka. In 2007, he became the captain of the ODI team before taking over in all formats by 2008.",
    "In the Indian Premier League (IPL), Dhoni plays for Chennai Super Kings (CSK), leading them to the final on ten occasions and winning it five times (2010, 2011, 2018, 2021 and 2023) jointly sharing this record with Rohit Sharma.",
    "The Indian ODI team in the early 2000s saw Rahul Dravid as the wicket-keeper to ensure that the wicket-keeper spot didn't lack in batting talent and also tried other wicket-keeper/batsmen like Parthiv Patel and Dinesh Karthik",
    "Dhoni played his last series during India's tour of Australia in December 2014. Following the third Test in Melbourne, Dhoni announced his retirement from the format."
]

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings=model.encode(docs).astype("float32")

In [ ]:
doc_embeddings.shape

In [ ]:
dimension=doc_embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
query="Tell me about Ms Dhoni"
query_embeddings=model.encode([query]).astype("float32")

top_k=2
distances,indices=index.search(query_embeddings,top_k)

retrieval_chuncks=[docs[i] for i in indices[0]]
print("Retrieval chuncks")
for chunck in retrieval_chuncks:
    print("-",chunck)

In [ ]:
query="Tell me about Rohit Sharma."
query_embeddings=model.encode([query]).astype("float32")

top_k=2
distances,indices=index.search(query_embeddings,top_k)

retrieval_chuncks=[docs[i] for i in indices[0]]
print("Retrieval chuncks")
for chunck in retrieval_chuncks:
    print("-",chunck)

In [ ]:
prompt=f"""
Answer the query based on the given context below.
context:{retrieval_chuncks}
query:{query}
answer formate: it must be polite and prefestional
"""

client = Groq(api_key=groq_api_key)

completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ],temperature=0.2
)

print(completion.choices[0].message.content)